# Годовые зависимости

Источник: `./data/{Месяц}/merged_with_mode.csv`. Все графики строятся без строк с пустым `PWR_MODE`.


In [1]:
import gc
import re
import os
from dotenv import load_dotenv
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


sns.set_theme(style="whitegrid")

DATA_DIR = Path("data")
OUT_DIR = Path("charts") / "part2_yearly_relationships"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MONTHS_RU = [
    "Январь", "Февраль", "Март", "Апрель", "Май", "Июнь",
    "Июль", "Август", "Сентябрь", "Октябрь", "Ноябрь", "Декабрь",
]

PWR_MODE_COL = "PWR_MODE"
POWER_COL = os.getenv("POWER_COL", "")

axial_cols_raw = os.getenv("AXIAL_COLS", "")
AXIAL_COLS = [col.strip() for col in axial_cols_raw.split(",")] if axial_cols_raw else []

load_dotenv()
vib_cols_raw = os.getenv("VIB_COLS", "")
VIB_COLS = [col.strip() for col in vib_cols_raw.split(",")] if vib_cols_raw else []
VIB_ALIASES = [f"vib_{i:02d}" for i in range(1, len(VIB_COLS) + 1)]
VIB_LABELS = dict(zip(VIB_COLS, VIB_ALIASES))


def display_name(col: str) -> str:
    return VIB_LABELS.get(col, col)

atr_cols_raw = os.getenv("ATR_COLS", "")
ATR_COLS = [col.strip() for col in atr_cols_raw.split(",")] if atr_cols_raw else []

t_cyl_cols_raw = os.getenv("T_CYL_COLS", "")
T_CYL_COLS = [col.strip() for col in t_cyl_cols_raw.split(",")] if t_cyl_cols_raw else []

def _sanitize_filename(name: str, max_len: int = 180) -> str:
    s = re.sub(r"[<>:\"/\\|?*\n\r\t]", "_", name)
    s = re.sub(r"\s+", " ", s).strip()
    if len(s) > max_len:
        s = s[:max_len].rstrip()
    return s


def _clean_power_mode(series: pd.Series) -> pd.Series:
    cleaned = series.astype("string").str.strip()
    invalid = cleaned.isna() | cleaned.eq("") | cleaned.str.lower().isin({"nan", "none", "null", "nat"})
    return cleaned.mask(invalid)


def _read_month_csv(month: str, usecols: list[str]) -> pd.DataFrame:
    path = DATA_DIR / month / "merged_with_mode.csv"
    if not path.exists():
        raise FileNotFoundError(f"Не найден файл: {path.as_posix()}")
    return pd.read_csv(
        path,
        sep=",",
        decimal=".",
        encoding="utf-8",
        low_memory=False,
        usecols=usecols,
    )


def load_year_data(needed_cols: list[str]) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    needed_cols = list(dict.fromkeys(needed_cols))
    usecols = list(dict.fromkeys([PWR_MODE_COL, *needed_cols]))

    for month in MONTHS_RU:
        path = DATA_DIR / month / "merged_with_mode.csv"
        if not path.exists():
            continue
        df = _read_month_csv(month, usecols=usecols)
        df[PWR_MODE_COL] = _clean_power_mode(df[PWR_MODE_COL])
        df = df.dropna(subset=[PWR_MODE_COL])
        frames.append(df[needed_cols])

    if not frames:
        raise RuntimeError("Не найдено ни одного `./data/{Месяц}/merged_with_mode.csv` для загрузки.")

    return pd.concat(frames, ignore_index=True)


def drop_outliers_central_interval(
    d: pd.DataFrame,
    x_col: str,
    y_col: str,
    interval: float = 0.99,
) -> pd.DataFrame:
    if not (0.0 < interval <= 1.0):
        raise ValueError("interval должен быть в (0; 1]. Например 0.99.")
    alpha = (1.0 - interval) / 2.0
    ql, qh = alpha, 1.0 - alpha

    dd = d.copy()
    dd[x_col] = pd.to_numeric(dd[x_col], errors="coerce")
    dd[y_col] = pd.to_numeric(dd[y_col], errors="coerce")
    dd = dd.dropna(subset=[x_col, y_col])
    if dd.empty:
        return dd

    x_low, x_high = dd[x_col].quantile(ql), dd[x_col].quantile(qh)
    y_low, y_high = dd[y_col].quantile(ql), dd[y_col].quantile(qh)
    mask = dd[x_col].between(x_low, x_high, inclusive="both") & dd[y_col].between(y_low, y_high, inclusive="both")
    return dd.loc[mask]


def plot_yearly_xy(
    year_data: pd.DataFrame,
    x_col: str,
    y_col: str,
    out_dir: Path,
    title_prefix: str,
    outlier_interval: float = 0.99,
    style: str = "scatter",  # scatter | hexbin
    fig_size: tuple[int, int] = (11, 7),
    dpi: int = 170,
) -> Path:
    d = year_data[[x_col, y_col]].copy()
    d = drop_outliers_central_interval(d, x_col=x_col, y_col=y_col, interval=outlier_interval)

    fig, ax = plt.subplots(figsize=fig_size, dpi=dpi)
    if d.empty:
        ax.text(0.5, 0.5, "нет данных", ha="center", va="center", transform=ax.transAxes)
    elif style == "hexbin":
        ax.hexbin(d[x_col], d[y_col], gridsize=55, mincnt=1, cmap="viridis", linewidths=0)
    else:
        ax.scatter(d[x_col], d[y_col], s=6, alpha=0.2, linewidths=0)

    ax.grid(True, linewidth=0.4, alpha=0.3)
    x_label = "pwr_mw"
    y_label = display_name(y_col)
    ax.set_xlabel("pwr_mw")
    ax.set_ylabel(y_label)
    ax.set_title(
        f"X: pwr_mw\nY: {y_label}\n"
        f"центральный интервал: {int(outlier_interval * 100)}%, точек: {len(d):,}"
    )

    plt.tight_layout()
    out_file = out_dir / f"{_sanitize_filename(title_prefix)}__X_{_sanitize_filename(x_label)}__Y_{_sanitize_filename(y_label)}.png"
    plt.savefig(out_file, bbox_inches="tight")
    plt.close(fig)
    return out_file

## Осевой сдвиг от мощности


In [ ]:
axial_out = OUT_DIR / "axial_vs_power"
axial_out.mkdir(parents=True, exist_ok=True)

year_axial = load_year_data([POWER_COL, *AXIAL_COLS])

axial_files: list[Path] = []
for y_col in AXIAL_COLS:
    axial_files.append(
        plot_yearly_xy(
            year_data=year_axial,
            x_col=POWER_COL,
            y_col=y_col,
            out_dir=axial_out,
            title_prefix="Осевой сдвиг от мощности",
        )
    )

del year_axial
gc.collect()

pd.DataFrame({"out": [p.as_posix() for p in axial_files]})


## Вибрация от мощности


In [ ]:
vib_out = OUT_DIR / "vibration_vs_power"
vib_out.mkdir(parents=True, exist_ok=True)

year_vib = load_year_data([POWER_COL, *VIB_COLS])

vib_files: list[Path] = []
for y_col in VIB_COLS:
    vib_files.append(
        plot_yearly_xy(
            year_data=year_vib,
            x_col=POWER_COL,
            y_col=y_col,
            out_dir=vib_out,
            title_prefix="Вибрация от мощности",
        )
    )

del year_vib
gc.collect()

pd.DataFrame({"out": [p.as_posix() for p in vib_files]})


## АТР от температуры цилиндра


In [ ]:
atr_t_out = OUT_DIR / "atr_vs_cylinder_temperature"
atr_t_out.mkdir(parents=True, exist_ok=True)

year_atr_t = load_year_data([*ATR_COLS, *T_CYL_COLS])

atr_t_files: list[Path] = []
for atr_col in ATR_COLS:
    for t_col in T_CYL_COLS:
        atr_t_files.append(
            plot_yearly_xy(
                year_data=year_atr_t,
                x_col=t_col,
                y_col=atr_col,
                out_dir=atr_t_out,
                title_prefix="АТР от температуры цилиндра",
            )
        )

del year_atr_t
gc.collect()

pd.DataFrame({"out": [p.as_posix() for p in atr_t_files]})
